In [1]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
# Load raw datasets
train_transaction = pd.read_csv("../data/raw/train_transaction.csv")
train_identity = pd.read_csv("../data/raw/train_identity.csv")

print("Transaction shape:", train_transaction.shape)
print("Identity shape:", train_identity.shape)

Transaction shape: (590540, 394)
Identity shape: (144233, 41)


In [2]:
from tqdm import tqdm

for i in tqdm(range(100)):
    # your operation here
    pass

100%|██████████| 100/100 [00:00<00:00, 1671037.45it/s]


In [3]:
# Merge transaction data with identity information
df = train_transaction.merge(
    train_identity,
    on="TransactionID",
    how="left"
)

print("Merged dataset shape:", df.shape)

print("\nTransactions retained:", len(df))

print(
    "Identity information available:",
    df["id_01"].notna().sum()
)

Merged dataset shape: (590540, 434)

Transactions retained: 590540
Identity information available: 144233


In [4]:
# Verify transaction-level uniqueness after merging
total_rows = len(df)
unique_transactions = df["TransactionID"].nunique()
duplicate_rows = df["TransactionID"].duplicated().sum()

print("Total rows:", total_rows)
print("Unique TransactionIDs:", unique_transactions)
print("Duplicate TransactionID rows:", duplicate_rows)

Total rows: 590540
Unique TransactionIDs: 590540
Duplicate TransactionID rows: 0


In [5]:
# Define target and remove the transaction identifier from model inputs

target = "isFraud"

X = df.drop(columns=["TransactionID", target])
y = df[target]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

print("\nTarget proportion:")
print(y.value_counts(normalize=True))

Feature matrix shape: (590540, 432)
Target shape: (590540,)

Target distribution:
isFraud
0    569877
1     20663
Name: count, dtype: int64

Target proportion:
isFraud
0    0.96501
1    0.03499
Name: proportion, dtype: float64


In [6]:
# Create time-based features from TransactionDT
# TransactionDT is measured in seconds from a reference point.

df["time_days"] = df["TransactionDT"] // (24 * 60 * 60)

df["time_week"] = df["TransactionDT"] // (7 * 24 * 60 * 60)

df["time_hour"] = (df["TransactionDT"] // (60 * 60)) % 24

df["time_dayofweek"] = (df["TransactionDT"] // (24 * 60 * 60)) % 7

print("Time features created:")
print([
    "time_days",
    "time_week",
    "time_hour",
    "time_dayofweek"
])

print("\nSample:")
print(
    df[
        [
            "TransactionDT",
            "time_days",
            "time_week",
            "time_hour",
            "time_dayofweek"
        ]
    ].head()
)

Time features created:
['time_days', 'time_week', 'time_hour', 'time_dayofweek']

Sample:
   TransactionDT  time_days  time_week  time_hour  time_dayofweek
0          86400          1          0          0               1
1          86401          1          0          0               1
2          86469          1          0          0               1
3          86499          1          0          0               1
4          86506          1          0          0               1


In [7]:
# Validate engineered time features

print("Time feature ranges:\n")

for col in [
    "time_days",
    "time_week",
    "time_hour",
    "time_dayofweek"
]:
    print(
        f"{col}:",
        "min =", df[col].min(),
        "| max =", df[col].max(),
        "| unique =", df[col].nunique()
    )

Time feature ranges:

time_days: min = 1 | max = 182 | unique = 182
time_week: min = 0 | max = 26 | unique = 27
time_hour: min = 0 | max = 23 | unique = 24
time_dayofweek: min = 0 | max = 6 | unique = 7


In [8]:
# Create log-transformed transaction amount
df["log_TransactionAmt"] = np.log1p(df["TransactionAmt"])

print("Original TransactionAmt:")
print(df["TransactionAmt"].describe())

print("\nLog-transformed TransactionAmt:")
print(df["log_TransactionAmt"].describe())

print("\nSample:")
print(
    df[
        ["TransactionAmt", "log_TransactionAmt"]
    ].head()
)

Original TransactionAmt:
count    590540.000000
mean        135.027176
std         239.162522
min           0.251000
25%          43.321000
50%          68.769000
75%         125.000000
max       31937.391000
Name: TransactionAmt, dtype: float64

Log-transformed TransactionAmt:
count    590540.000000
mean          4.382960
std           0.937183
min           0.223943
25%           3.791459
50%           4.245190
75%           4.836282
max          10.371564
Name: log_TransactionAmt, dtype: float64

Sample:
   TransactionAmt  log_TransactionAmt
0            68.5            4.241327
1            29.0            3.401197
2            59.0            4.094345
3            50.0            3.931826
4            50.0            3.931826


In [9]:
# Create an identity-availability indicator
df["has_identity"] = df["id_01"].notna().astype(int)

print("Identity availability:")
print(df["has_identity"].value_counts())

print("\nIdentity availability proportion:")
print(df["has_identity"].value_counts(normalize=True))

Identity availability:
has_identity
0    446307
1    144233
Name: count, dtype: int64

Identity availability proportion:
has_identity
0    0.755761
1    0.244239
Name: proportion, dtype: float64


## Step 9 — Missingness Features

Fraud datasets often contain substantial missing data. In this dataset, missingness
may itself contain useful information because the availability of certain
transaction or identity attributes can differ between transactions.

Instead of immediately removing columns with missing values, we will first
quantify the amount of missing information available for each transaction.

We will create:

- `missing_count`: number of missing feature values for a transaction
- `missing_ratio`: proportion of feature values that are missing

These features will be evaluated as candidate predictors during later
leakage-safe model validation.

In [10]:
# Create overall missingness features
# Exclude the target and TransactionID from the calculation.

feature_columns = [
    col for col in df.columns
    if col not in ["isFraud", "TransactionID"]
]

df["missing_count"] = df[feature_columns].isna().sum(axis=1)

df["missing_ratio"] = (
    df["missing_count"] / len(feature_columns)
)

print("Missingness features created:")
print(
    df[
        ["missing_count", "missing_ratio"]
    ].describe()
)

print("\nSample:")
print(
    df[
        ["missing_count", "missing_ratio"]
    ].head()
)

Missingness features created:
       missing_count  missing_ratio
count  590540.000000  590540.000000
mean      195.622774       0.446627
std        49.035963       0.111954
min        24.000000       0.054795
25%       208.000000       0.474886
50%       211.000000       0.481735
75%       229.000000       0.522831
max       340.000000       0.776256

Sample:
   missing_count  missing_ratio
0            234       0.534247
1            230       0.525114
2            211       0.481735
3            227       0.518265
4            137       0.312785


# Step 10 — Missingness vs Fraud

In [11]:
missingness_by_fraud = (
    df.groupby("isFraud")[["missing_count", "missing_ratio"]]
      .mean()
)

print(missingness_by_fraud)

         missing_count  missing_ratio
isFraud                              
0           196.852849       0.449436
1           161.697817       0.369173


# Step 11 — Identity Missingness

In [12]:
identity_columns = train_identity.columns.drop("TransactionID")

df["identity_missing_count"] = df[identity_columns].isna().sum(axis=1)

print(df["identity_missing_count"].describe())

count    590540.000000
mean         33.793455
std          11.221650
min           0.000000
25%          40.000000
50%          40.000000
75%          40.000000
max          40.000000
Name: identity_missing_count, dtype: float64


# Step 12 — Categorical Feature Identification

In [13]:
categorical_features = df.select_dtypes(include="object").columns.tolist()

print("Number of categorical features:", len(categorical_features))
print("\nCategorical features:")
print(categorical_features)

Number of categorical features: 31

Categorical features:
['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']


# Step 13 — Categorical Feature Cardinality Check
Categorical features need different handling depending on how many unique categories they contain.

- Low cardinality → generally suitable for one-hot encoding.
- Moderate cardinality → may need careful encoding.
- High cardinality → one-hot encoding can create too many columns and memory issues.

DeviceInfo is particularly important because we already know it has many unique values.


In [14]:
# Check the number of unique values in each categorical feature

categorical_cardinality = (
    df[categorical_features]
    .nunique(dropna=False)
    .sort_values(ascending=False)
)

print("Categorical feature cardinality:")
print(categorical_cardinality)

Categorical feature cardinality:
DeviceInfo       1787
id_33             261
id_31             131
id_30              76
R_emaildomain      61
P_emaildomain      60
card4               5
id_34               5
ProductCD           5
card6               5
id_15               4
M4                  4
id_23               4
M3                  3
DeviceType          3
id_38               3
id_37               3
id_36               3
id_35               3
M1                  3
M2                  3
M5                  3
M6                  3
id_28               3
id_27               3
id_16               3
id_12               3
M9                  3
M8                  3
M7                  3
id_29               3
dtype: int64


# Step 14 — Rare Category Analysis

In [15]:
# Check the most frequent categories and their frequency
# for selected higher-cardinality categorical features.

high_cardinality_features = [
    "DeviceInfo",
    "id_33",
    "id_31",
    "id_30",
    "P_emaildomain",
    "R_emaildomain"
]

for col in high_cardinality_features:
    print(f"\n{'=' * 60}")
    print(f"{col} — Top 10 categories")
    print(df[col].value_counts(dropna=False).head(10))


DeviceInfo — Top 10 categories
DeviceInfo
NaN                      471874
Windows                   47722
iOS Device                19782
MacOS                     12573
Trident/7.0                7440
rv:11.0                    1901
rv:57.0                     962
SM-J700M Build/MMB29K       549
SM-G610M Build/MMB29K       461
SM-G531H Build/LMY48B       410
Name: count, dtype: int64

id_33 — Top 10 categories
id_33
NaN          517251
1920x1080     16874
1366x768       8605
1334x750       6447
2208x1242      4900
1440x900       4384
1600x900       3510
2048x1536      3482
1280x800       2149
2560x1600      2093
Name: count, dtype: int64

id_31 — Top 10 categories
id_31
NaN                        450258
chrome 63.0                 22000
mobile safari 11.0          13423
mobile safari generic       11474
ie 11.0 for desktop          9030
safari generic               8195
chrome 62.0                  7182
chrome 65.0                  6871
chrome 64.0                  6711
chrome 63.0 f

In [16]:
# Analyze how many categories are genuinely rare
# in the higher-cardinality categorical features.

for col in high_cardinality_features:
    value_counts = df[col].value_counts(dropna=False)

    rare_1 = (value_counts <= 1).sum()
    rare_5 = (value_counts <= 5).sum()
    rare_10 = (value_counts <= 10).sum()

    print(f"\n{col}")
    print(f"Total categories: {len(value_counts)}")
    print(f"Categories appearing <= 1 time: {rare_1}")
    print(f"Categories appearing <= 5 times: {rare_5}")
    print(f"Categories appearing <= 10 times: {rare_10}")


DeviceInfo
Total categories: 1787
Categories appearing <= 1 time: 440
Categories appearing <= 5 times: 1018
Categories appearing <= 10 times: 1304

id_33
Total categories: 261
Categories appearing <= 1 time: 85
Categories appearing <= 5 times: 161
Categories appearing <= 10 times: 184

id_31
Total categories: 131
Categories appearing <= 1 time: 15
Categories appearing <= 5 times: 23
Categories appearing <= 10 times: 28

id_30
Total categories: 76
Categories appearing <= 1 time: 1
Categories appearing <= 5 times: 4
Categories appearing <= 10 times: 5

P_emaildomain
Total categories: 60
Categories appearing <= 1 time: 0
Categories appearing <= 5 times: 0
Categories appearing <= 10 times: 0

R_emaildomain
Total categories: 61
Categories appearing <= 1 time: 0
Categories appearing <= 5 times: 0
Categories appearing <= 10 times: 2


### Decision — Categorical Feature Handling

| Feature group | Features | Decision |
|---|---|---|
| High cardinality | `DeviceInfo`, `id_33`, `id_31` | Rare-category grouping; frequency-based representation considered |
| Moderate cardinality | `id_30`, `P_emaildomain`, `R_emaildomain` | Retain; standard categorical encoding |
| Low cardinality — Transaction/Card | `ProductCD`, `card4`, `card6` | Retain; standard categorical encoding |
| Low cardinality — M features | `M1`–`M9` | Retain; standard categorical encoding |
| Low cardinality — Identity | `id_12`, `id_15`, `id_16`, `id_23`, `id_27`, `id_28`, `id_29`, `id_34`–`id_38` | Retain; standard categorical encoding |
| Device category | `DeviceType` | Retain; standard categorical encoding |
| Missing values | All categorical features | Preserve missingness through explicit handling |
| Target encoding | All categorical features | Not used on the complete dataset to prevent target leakage |
| Frequency encoding | Selected high-cardinality features | Fit on training data only and apply to validation/future data |

**Overall Decision:**  
Categorical features will be handled according to their cardinality and
frequency rather than using one universal encoding strategy. Final feature
usefulness will be determined through leakage-safe temporal model validation.

# Step 15 — Numerical Feature Quality Analysis

In [17]:
# Identify numerical features
numerical_features = df.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

print("Number of numerical features:", len(numerical_features))

# Build a numerical feature-quality summary
numerical_quality = pd.DataFrame({
    "dtype": df[numerical_features].dtypes,
    "missing_count": df[numerical_features].isna().sum(),
    "missing_pct": df[numerical_features].isna().mean() * 100,
    "unique_values": df[numerical_features].nunique(dropna=True)
})

# Sort by missingness
numerical_quality = numerical_quality.sort_values(
    "missing_pct",
    ascending=False
)

print("\nTop 20 numerical features by missing percentage:")
print(numerical_quality.head(20))

print("\nNumerical features with <= 1 unique non-missing value:")
print(
    numerical_quality[
        numerical_quality["unique_values"] <= 1
    ]
)

Number of numerical features: 412

Top 20 numerical features by missing percentage:
         dtype  missing_count  missing_pct  unique_values
id_24  float64         585793    99.196159             12
id_25  float64         585408    99.130965            341
id_08  float64         585385    99.127070             94
id_07  float64         585385    99.127070             84
id_21  float64         585381    99.126393            490
id_26  float64         585377    99.125715             95
id_22  float64         585371    99.124699             25
dist2  float64         552913    93.628374           1751
D7     float64         551623    93.409930            597
id_18  float64         545427    92.360721             18
D13    float64         528588    89.509263            577
D14    float64         528353    89.469469            802
D12    float64         525823    89.041047            635
id_03  float64         524216    88.768923             24
id_04  float64         524216    88.768923    

# Step 16 — Numerical Feature Grouping

In [18]:
# Group numerical features by their feature families

numerical_groups = {
    "Transaction": [
        col for col in numerical_features
        if col in ["TransactionID", "TransactionDT", "TransactionAmt"]
    ],
    "Card": [
        col for col in numerical_features
        if col.startswith("card")
    ],
    "Address": [
        col for col in numerical_features
        if col.startswith("addr")
    ],
    "Distance": [
        col for col in numerical_features
        if col.startswith("dist")
    ],
    "C": [
        col for col in numerical_features
        if col.startswith("C")
    ],
    "D": [
        col for col in numerical_features
        if col.startswith("D")
    ],
    "V": [
        col for col in numerical_features
        if col.startswith("V")
    ],
    "Identity": [
        col for col in numerical_features
        if col.startswith("id_")
    ]
}

print("Numerical feature groups:\n")

for group, features in numerical_groups.items():
    print(f"{group}: {len(features)}")

Numerical feature groups:

Transaction: 3
Card: 4
Address: 2
Distance: 2
C: 14
D: 15
V: 339
Identity: 23


# Step 17 — Identify Near-Constant Numerical Features

## Numerical Feature Quality — Near-Constant Features

Numerical features are examined for extremely low variability before modeling.

A feature with very little variation provides limited discriminatory information and
may increase model complexity without adding meaningful predictive value.

Rather than removing features solely because of high missingness, this step focuses
on whether the observed values themselves contain sufficient variation.

In [19]:
# Identify near-constant numerical features
# We examine the most frequent value in each feature.

near_constant_summary = []

for col in numerical_features:
    value_counts = df[col].value_counts(dropna=True)

    if len(value_counts) == 0:
        continue

    most_common_count = value_counts.iloc[0]
    non_missing_count = df[col].notna().sum()

    most_common_ratio = (
        most_common_count / non_missing_count
    )

    near_constant_summary.append({
        "feature": col,
        "unique_values": df[col].nunique(dropna=True),
        "most_common_ratio": most_common_ratio,
        "missing_pct": df[col].isna().mean() * 100
    })

near_constant_summary = pd.DataFrame(
    near_constant_summary
).sort_values(
    "most_common_ratio",
    ascending=False
)

print("Top 20 numerical features by most-common-value ratio:")
print(
    near_constant_summary.head(20)
)

print("\nFeatures where one value represents >= 99% of non-missing observations:")
print(
    near_constant_summary[
        near_constant_summary["most_common_ratio"] >= 0.99
    ]
)

Top 20 numerical features by most-common-value ratio:
    feature  unique_values  most_common_ratio  missing_pct
345    V305              2           0.999993     0.002032
41       V1              2           0.999945    47.293494
281    V241              5           0.999724    77.913435
105     V65              2           0.999663    13.055170
147    V107              2           0.999580     0.053172
54      V14              2           0.999500    12.881939
108     V68              3           0.999482    13.055170
81      V41              2           0.999269    28.612626
128     V88              2           0.999246    15.098723
67      V27              4           0.999240    12.881939
68      V28              4           0.999234    12.881939
280    V240              6           0.999233    77.913435
129     V89              3           0.999158    15.098723
157    V117              4           0.998768     0.053172
159    V119              4           0.998685     0.053172
15

## Step 18 -  Numerical Feature Quality — Missingness Analysis

Missing values are not automatically treated as unusable in the IEEE-CIS dataset.

Because the availability of transaction and identity information can vary across
transactions, the presence or absence of a feature may itself contain predictive
information.

For each numerical feature, the fraud rate among transactions where the feature
is missing is compared with the fraud rate where the feature is observed.

This analysis is descriptive and does not imply that missingness causes fraud.

In [20]:
# Compare fraud rates for missing vs observed values
# across numerical features.

missingness_analysis = []

overall_fraud_rate = df["isFraud"].mean()

for col in numerical_features:
    missing_mask = df[col].isna()
    observed_mask = df[col].notna()

    missing_count = missing_mask.sum()
    observed_count = observed_mask.sum()

    if missing_count > 0:
        fraud_rate_missing = df.loc[
            missing_mask, "isFraud"
        ].mean()
    else:
        fraud_rate_missing = np.nan

    if observed_count > 0:
        fraud_rate_observed = df.loc[
            observed_mask, "isFraud"
        ].mean()
    else:
        fraud_rate_observed = np.nan

    missingness_analysis.append({
        "feature": col,
        "missing_pct": missing_mask.mean() * 100,
        "missing_count": missing_count,
        "fraud_rate_missing": fraud_rate_missing,
        "fraud_rate_observed": fraud_rate_observed
    })

missingness_analysis = pd.DataFrame(
    missingness_analysis
)

missingness_analysis["fraud_rate_difference"] = (
    missingness_analysis["fraud_rate_missing"]
    - missingness_analysis["fraud_rate_observed"]
)

# Largest absolute differences first
missingness_analysis["absolute_difference"] = (
    missingness_analysis["fraud_rate_difference"].abs()
)

missingness_analysis = missingness_analysis.sort_values(
    "absolute_difference",
    ascending=False
)

print("Top 20 numerical features by missing-vs-observed fraud-rate difference:")
print(
    missingness_analysis.head(20)
)

Top 20 numerical features by missing-vs-observed fraud-rate difference:
    feature  missing_pct  missing_count  fraud_rate_missing  \
319    V279     0.002032             12            0.166667   
325    V285     0.002032             12            0.166667   
327    V287     0.002032             12            0.166667   
330    V290     0.002032             12            0.166667   
332    V292     0.002032             12            0.166667   
333    V293     0.002032             12            0.166667   
334    V294     0.002032             12            0.166667   
335    V295     0.002032             12            0.166667   
337    V297     0.002032             12            0.166667   
338    V298     0.002032             12            0.166667   
339    V299     0.002032             12            0.166667   
342    V302     0.002032             12            0.166667   
343    V303     0.002032             12            0.166667   
344    V304     0.002032             12       

## Step 19 - Numerical Feature Quality — Low-Cardinality Numerical Features

Some numerical variables contain only a small number of distinct values.

Although stored as numerical data types, such variables may behave more like
categorical or discrete features.

These features are identified before preprocessing so that they are not
automatically treated as continuous measurements without inspection.

In [21]:
# Identify numerical features with low cardinality
# Exclude the target variable from feature analysis.

low_cardinality_numeric = (
    numerical_quality[
        (numerical_quality["unique_values"] <= 10)
        & (numerical_quality.index != "isFraud")
    ]
    .sort_values("unique_values")
)

print(
    "Numerical features with 10 or fewer unique non-missing values:"
)

print(
    low_cardinality_numeric.to_string()
)

Numerical features with 10 or fewer unique non-missing values:
                  dtype  missing_count  missing_pct  unique_values
has_identity      int64              0     0.000000              2
V88             float64          89164    15.098723              2
V1              float64         279287    47.293494              2
V65             float64          77096    13.055170              2
V107            float64            314     0.053172              2
V14             float64          76073    12.881939              2
V305            float64             12     0.002032              2
V41             float64         168969    28.612626              2
V68             float64          77096    13.055170              3
V94             float64          89164    15.098723              3
V89             float64          89164    15.098723              3
V117            float64            314     0.053172              4
V118            float64            314     0.053172              4

## Step 20 - Feature Engineering — Final Feature Inventory

Before preprocessing, all candidate features are grouped according to their
role in the fraud detection pipeline.

The target variable and transaction identifier are excluded from the model.

Transaction time is retained as the primary temporal signal, while derived
time features provide additional temporal information.

Transaction amount is retained in both its original and log-transformed form.

Categorical variables will be encoded according to their cardinality, while
numerical variables will remain numerical unless later validation indicates
otherwise.

Features that are identifiers, targets, or EDA-only helper variables will not
be used as model inputs.

In [22]:
# Final feature inventory

target = "isFraud"
identifier = "TransactionID"

# EDA-only helper columns that should not enter the model directly
eda_only_columns = [
    "amount_range"
]

# Candidate model features
model_features = [
    col for col in df.columns
    if col not in [
        target,
        identifier,
        *eda_only_columns
    ]
]

# Separate categorical and numerical features
categorical_model_features = df[
    model_features
].select_dtypes(include="object").columns.tolist()

numerical_model_features = df[
    model_features
].select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Total candidate model features:", len(model_features))

print("\nCategorical features:", len(categorical_model_features))
print(categorical_model_features)

print("\nNumerical features:", len(numerical_model_features))
print(numerical_model_features[:30])

print("\nExcluded from modeling:")
print("Target:", target)
print("Identifier:", identifier)
print("EDA-only:", eda_only_columns)

Total candidate model features: 441

Categorical features: 31
['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']

Numerical features: 410
['TransactionDT', 'TransactionAmt', 'card1', 'card2', 'card3', 'card5', 'addr1', 'addr2', 'dist1', 'dist2', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6']

Excluded from modeling:
Target: isFraud
Identifier: TransactionID
EDA-only: ['amount_range']


# Step 21 — Leakage Check Before Preprocessing
## Leakage Check

The fraud detection model must only use information that would be available
at the time a transaction is evaluated.

Potential leakage can arise from target-derived variables, transaction
identifiers, post-transaction information, or features constructed using
future observations.

The current feature set is therefore checked for:

- Explicit fraud-related column names
- Target and identifier leakage
- EDA-only helper variables
- Suspicious feature names that require further investigation

Features that pass this structural check will still be evaluated using
temporal validation because leakage can also occur during preprocessing and
feature engineering.

In [23]:
# Step 21 — Structural leakage check

# 1. Check for column names containing fraud-related terms
suspicious_name_terms = [
    "fraud",
    "target",
    "label"
]

suspicious_columns = [
    col for col in model_features
    if any(term in col.lower() for term in suspicious_name_terms)
]

print("Columns with suspicious target-related names:")
print(suspicious_columns)

# 2. Verify target and identifier are not model features
print("\nTarget included in model features:", target in model_features)
print("Identifier included in model features:", identifier in model_features)

# 3. Check EDA-only columns
eda_in_model = [
    col for col in eda_only_columns
    if col in model_features
]

print("\nEDA-only columns included in model features:")
print(eda_in_model)

# 4. Check for duplicate feature names
duplicate_feature_names = (
    pd.Series(model_features)
    .duplicated()
    .sum()
)

print("\nDuplicate feature names:", duplicate_feature_names)

# 5. Final structural leakage status
if (
    len(suspicious_columns) == 0
    and target not in model_features
    and identifier not in model_features
    and len(eda_in_model) == 0
    and duplicate_feature_names == 0
):
    print("\nStructural leakage check: PASSED")
else:
    print("\nStructural leakage check: REVIEW REQUIRED")

Columns with suspicious target-related names:
[]

Target included in model features: False
Identifier included in model features: False

EDA-only columns included in model features:
[]

Duplicate feature names: 0

Structural leakage check: PASSED


# Step 22 — Temporal Train/Validation Split.

## Temporal Train / Validation Split

Fraud patterns can change over time, so model evaluation should reflect the
real-world prediction scenario.

The model will be trained on earlier transactions and evaluated on later
transactions.

An 80/20 chronological split will be used:

- First 80% of transactions by `TransactionDT` → Training set
- Final 20% of transactions by `TransactionDT` → Validation set

No preprocessing parameters, categorical encodings, or frequency statistics
will be learned from the validation period.

This prevents future information from influencing model development.

In [24]:
# Step 22 — Temporal train/validation split

# Sort transactions chronologically
df = df.sort_values("TransactionDT").reset_index(drop=True)

# Determine the 80% chronological split point
split_index = int(len(df) * 0.80)

# Create train and validation datasets
train_df = df.iloc[:split_index].copy()
valid_df = df.iloc[split_index:].copy()

print("Total transactions:", len(df))

print("\nTraining set:")
print("Rows:", len(train_df))
print(
    "TransactionDT range:",
    train_df["TransactionDT"].min(),
    "to",
    train_df["TransactionDT"].max()
)

print("\nValidation set:")
print("Rows:", len(valid_df))
print(
    "TransactionDT range:",
    valid_df["TransactionDT"].min(),
    "to",
    valid_df["TransactionDT"].max()
)

print("\nTraining fraud rate:", train_df["isFraud"].mean())
print("Validation fraud rate:", valid_df["isFraud"].mean())

Total transactions: 590540

Training set:
Rows: 472432
TransactionDT range: 86400 to 12192842

Validation set:
Rows: 118108
TransactionDT range: 12192900 to 15811131

Training fraud rate: 0.03513521522674162
Validation fraud rate: 0.034409184813899145


# Step 23A — Compare category coverage
## Categorical Feature Coverage

Categorical features may contain values in the future validation period that
were not observed during the historical training period.

This is particularly important for high-cardinality variables such as
`DeviceInfo`, `id_31`, and `id_33`.

The training and validation category sets will therefore be compared before
encoding.

The preprocessing pipeline will be designed to handle unseen validation
categories without using validation information during fitting.

In [25]:
# Step 23A — Compare categorical values between train and validation

category_coverage = []

for col in categorical_model_features:

    train_categories = set(
        train_df[col].dropna().unique()
    )

    valid_categories = set(
        valid_df[col].dropna().unique()
    )

    unseen_categories = valid_categories - train_categories

    category_coverage.append({
        "feature": col,
        "train_categories": len(train_categories),
        "validation_categories": len(valid_categories),
        "unseen_in_validation": len(unseen_categories)
    })

category_coverage = pd.DataFrame(category_coverage)

category_coverage = category_coverage.sort_values(
    "unseen_in_validation",
    ascending=False
)

print(category_coverage.to_string(index=False))

      feature  train_categories  validation_categories  unseen_in_validation
   DeviceInfo              1639                    848                   147
        id_33               212                    162                    48
        id_31               110                    104                    20
        id_30                72                     70                     3
        id_16                 2                      2                     0
   DeviceType                 2                      2                     0
        id_38                 2                      2                     0
        id_37                 2                      2                     0
        id_36                 2                      2                     0
        id_35                 2                      2                     0
        id_34                 4                      3                     0
        id_29                 2                      2                     0

# Step 23B — Training Category Frequency Analysis

## Categorical Frequency Analysis

Categorical cardinality alone does not determine the appropriate encoding
strategy.

The frequency distribution of categories is also important because highly
fragmented categorical variables can produce very large sparse feature spaces
and may contain many categories that are rarely observed.

The training data will therefore be examined for category frequency
concentration, particularly for high-cardinality features such as
`DeviceInfo`, `id_31`, and `id_33`.

Frequency statistics are calculated using the training period only so that
validation information is not used during preprocessing decisions.

In [26]:
# Step 23B — Training category frequency analysis

high_cardinality_features = [
    "DeviceInfo",
    "id_33",
    "id_31"
]

for col in high_cardinality_features:

    print("\n" + "=" * 70)
    print(f"Feature: {col}")
    print("=" * 70)

    # Count category frequencies, including missing values
    frequencies = train_df[col].fillna("__MISSING__").value_counts()

    total_categories = len(frequencies)
    categories_once = (frequencies <= 1).sum()
    categories_5_or_less = (frequencies <= 5).sum()
    categories_10_or_less = (frequencies <= 10).sum()

    print(f"Total categories: {total_categories}")
    print(f"Categories appearing once: {categories_once}")
    print(f"Categories appearing <= 5 times: {categories_5_or_less}")
    print(f"Categories appearing <= 10 times: {categories_10_or_less}")

    print("\nTop 10 categories:")
    print(frequencies.head(10).to_string())


Feature: DeviceInfo
Total categories: 1640
Categories appearing once: 418
Categories appearing <= 5 times: 989
Categories appearing <= 10 times: 1229

Top 10 categories:
DeviceInfo
__MISSING__              372896
Windows                   39681
iOS Device                17083
MacOS                     10843
Trident/7.0                6453
rv:11.0                    1651
rv:57.0                     957
SM-J700M Build/MMB29K       441
SM-G610M Build/MMB29K       385
SM-G531H Build/LMY48B       341

Feature: id_33
Total categories: 213
Categories appearing once: 68
Categories appearing <= 5 times: 127
Categories appearing <= 10 times: 144

Top 10 categories:
id_33
__MISSING__    410619
1920x1080       14460
1366x768         6669
1334x750         5602
2208x1242        4166
1440x900         3792
2048x1536        3110
1600x900         3098
1280x800         1882
2560x1600        1818

Feature: id_31
Total categories: 111
Categories appearing once: 11
Categories appearing <= 5 times: 18
Categ

# Step 23C — Numerical preprocessing strategy
## Numerical Feature Quality Assessment

Numerical features are evaluated using the training period only before
preprocessing.

The assessment focuses on missingness and variation because features with
extreme missingness or negligible variation may add little useful information
while increasing model complexity.

Missing values will not be removed automatically because missingness itself
may contain predictive information in fraud detection.

The final preprocessing pipeline will therefore combine feature-quality
filtering with appropriate numerical imputation.

In [27]:
# Step 23C — Numerical feature quality assessment

numerical_features = [
    col for col in model_features
    if col in numerical_model_features
]

numerical_quality = []

for col in numerical_features:

    missing_count = train_df[col].isna().sum()
    missing_pct = missing_count / len(train_df) * 100

    non_missing = train_df[col].dropna()

    unique_values = non_missing.nunique()

    numerical_quality.append({
        "feature": col,
        "missing_count": missing_count,
        "missing_pct": missing_pct,
        "unique_values": unique_values
    })

numerical_quality = pd.DataFrame(numerical_quality)

print("Total numerical features:", len(numerical_quality))

print("\nHighly missing features (>90%):")
print(
    numerical_quality[
        numerical_quality["missing_pct"] > 90
    ]
    .sort_values("missing_pct", ascending=False)
    .head(20)
    .to_string(index=False)
)

print("\nNear-constant features (<=2 unique non-missing values):")
print(
    numerical_quality[
        numerical_quality["unique_values"] <= 2
    ]
    .sort_values("unique_values")
    .to_string(index=False)
)

print("\nMissingness summary:")
print(
    numerical_quality["missing_pct"]
    .describe()
    .to_string()
)

Total numerical features: 410

Highly missing features (>90%):
feature  missing_count  missing_pct  unique_values
  id_24         468518    99.171521             11
  id_25         468186    99.101246            317
  id_07         468171    99.098071             82
  id_08         468171    99.098071             93
  id_21         468169    99.097648            444
  id_26         468165    99.096801             89
  id_22         468159    99.095531             23
     D7         442488    93.661733            568
  dist2         440332    93.205371           1664
  id_18         435010    92.078860             18

Near-constant features (<=2 unique non-missing values):
     feature  missing_count  missing_pct  unique_values
          V1         245973    52.065271              2
         V14          66427    14.060648              2
         V41         139809    29.593465              2
         V65          66890    14.158651              2
         V88          77793    16.46649

# Step 23D — Duplicate and near-duplicate numerical features
## Numerical Redundancy Check

The numerical feature set contains many anonymized variables, so some
features may contain identical or effectively duplicated information.

Exact duplicate columns and perfectly correlated feature pairs are checked
before preprocessing.

This step is used only to identify redundant representations. Predictive
feature selection will be based on leakage-safe validation rather than
correlation alone.

In [28]:
# Step 23D — Check numerical redundancy

numeric_train = train_df[numerical_features]

# ---------------------------------------------------------
# 1. Exact duplicate columns
# ---------------------------------------------------------

duplicate_columns = []

columns = numeric_train.columns

for i in range(len(columns)):
    for j in range(i + 1, len(columns)):

        col1 = columns[i]
        col2 = columns[j]

        if numeric_train[col1].equals(numeric_train[col2]):
            duplicate_columns.append((col1, col2))

print("Exact duplicate column pairs:", len(duplicate_columns))

if duplicate_columns:
    print("\nDuplicate pairs:")
    for pair in duplicate_columns[:20]:
        print(pair)


# ---------------------------------------------------------
# 2. Perfect correlation
# ---------------------------------------------------------

correlation_matrix = numeric_train.corr()

perfect_pairs = []

for i in range(len(correlation_matrix.columns)):
    for j in range(i + 1, len(correlation_matrix.columns)):

        corr_value = correlation_matrix.iloc[i, j]

        if abs(corr_value) == 1:
            perfect_pairs.append(
                (
                    correlation_matrix.columns[i],
                    correlation_matrix.columns[j],
                    corr_value
                )
            )

print("\nPerfectly correlated pairs:", len(perfect_pairs))

if perfect_pairs:
    print("\nPerfectly correlated pairs:")
    for pair in perfect_pairs[:20]:
        print(pair)

Exact duplicate column pairs: 0

Perfectly correlated pairs: 1

Perfectly correlated pairs:
('D4', 'D12', 1.0)


## Step 23 — Preprocessing & Feature Quality Audit Results

| Step | Analysis | Key Result | Decision |
|---|---|---|---|
| 23A | Train–validation categorical coverage | `DeviceInfo`: 147 unseen categories; `id_33`: 48; `id_31`: 20 | Use train-fitted preprocessing with safe handling of unseen categories |
| 23B | `DeviceInfo` cardinality | 1,640 categories; 418 appear once; 989 appear ≤5 times; 1,229 appear ≤10 times | Frequency encoding |
| 23B | `id_33` cardinality | 213 categories; 68 appear once; 127 appear ≤5 times; 144 appear ≤10 times | Frequency encoding |
| 23B | `id_31` cardinality | 111 categories; 11 appear once; 18 appear ≤5 times; 25 appear ≤10 times | Rare grouping + one-hot encoding |
| 23C | Numerical feature count | 410 numerical candidate features | Retain initially; filter clear redundancy/near-constant issues |
| 23C | Numerical missingness | Mean: 43.0%; median: 47.5%; maximum: 99.17% | Do not automatically drop highly missing features |
| 23C | Highly missing numerical features | Several identity features exceed 99% missingness | Retain initially; evaluate through validation |
| 23C | Near-constant numerical features | `V1`, `V14`, `V41`, `V65`, `V88`, `V107`, `V305` have ≤2 unique values | Flag for removal |
| 23C | `has_identity` | Binary feature with 2 values | Keep — meaningful availability signal |
| 23D | Exact duplicate numerical columns | 0 duplicate pairs | No action |
| 23D | Perfectly correlated numerical features | `D4` and `D12`, correlation = 1.0 | Drop `D12`; retain `D4` |

### Final Preprocessing Strategy

- Low-cardinality categorical features → one-hot encoding
- `id_30` and `id_31` → rare-category grouping + one-hot encoding
- `DeviceInfo` and `id_33` → training-only frequency encoding
- Missing categorical values → retained as an explicit category
- Unseen validation categories → handled safely during transformation
- Numerical missing values → training-fitted median imputation
- Numerical features → scaling for Logistic Regression
- Highly missing numerical features → retained initially and evaluated through model validation
- Near-constant V-features → candidates for removal
- `D12` → removed because it is perfectly redundant with `D4`
- Target encoding and historical fraud-rate encoding → not used because of leakage risk

# Step 24 — Leakage-Safe Preprocessing

## Step 24A — Final Feature Groups

The audited feature set is divided into numerical and categorical preprocessing
groups before model development.

Redundant feature `D12` is removed because it is perfectly correlated with
`D4`.

High-cardinality categorical variables such as `DeviceInfo` and `id_33` will
use frequency encoding based only on the training period.

Moderate-cardinality variables `id_30` and `id_31` will use rare-category
grouping followed by one-hot encoding.

The remaining categorical variables will use one-hot encoding.

All preprocessing decisions will be learned from the training data only.

In [29]:
# Step 24A — Final feature groups before preprocessing

# Remove redundant numerical feature
redundant_features = ["D12"]

model_features_final = [
    col for col in model_features
    if col not in redundant_features
]

# High-cardinality categorical features
frequency_features = [
    "DeviceInfo",
    "id_33"
]

# Moderate-cardinality categorical features
rare_group_features = [
    "id_31",
    "id_30"
]

# Remaining categorical features → one-hot encoding
onehot_features = [
    col for col in categorical_model_features
    if col not in frequency_features + rare_group_features
]

# Final numerical features
final_numerical_features = [
    col for col in numerical_model_features
    if col not in redundant_features
]

print("Final candidate features:", len(model_features_final))
print("Numerical features:", len(final_numerical_features))
print("Frequency-encoded features:", len(frequency_features))
print("Rare-group + one-hot features:", len(rare_group_features))
print("Other one-hot features:", len(onehot_features))

print("\nFrequency-encoded:")
print(frequency_features)

print("\nRare-group + one-hot:")
print(rare_group_features)

Final candidate features: 440
Numerical features: 409
Frequency-encoded features: 2
Rare-group + one-hot features: 2
Other one-hot features: 27

Frequency-encoded:
['DeviceInfo', 'id_33']

Rare-group + one-hot:
['id_31', 'id_30']


## Step 24B — Training-Only Frequency Encoding

High-cardinality categorical variables are converted into numerical frequency
features to avoid creating excessively large sparse feature spaces.

The category frequencies are calculated using the training period only.

Validation categories that were not observed during training are assigned a
frequency of zero.

Missing values are treated as an explicit category so that missingness is
preserved as information.

The resulting frequency features are used in place of the original
high-cardinality categorical variables.

In [30]:
# Step 24B — Training-only frequency encoding

frequency_maps = {}

for col in frequency_features:

    # Treat missing values as an explicit category
    train_values = train_df[col].fillna("__MISSING__")

    # Calculate frequencies using TRAINING DATA ONLY
    frequency_map = train_values.value_counts(normalize=True)

    frequency_maps[col] = frequency_map

    print("\n" + "=" * 60)
    print(f"Feature: {col}")
    print("=" * 60)

    print("Training categories:", len(frequency_map))
    print("Minimum frequency:", frequency_map.min())
    print("Maximum frequency:", frequency_map.max())

    print("\nTop 5 frequencies:")
    print(frequency_map.head(5).to_string())


# Apply the training frequency mappings to both datasets

for col in frequency_features:

    train_values = train_df[col].fillna("__MISSING__")
    valid_values = valid_df[col].fillna("__MISSING__")

    train_df[f"{col}_freq"] = train_values.map(
        frequency_maps[col]
    )

    valid_df[f"{col}_freq"] = valid_values.map(
        frequency_maps[col]
    ).fillna(0)


print("\nFrequency encoding completed.")

print("\nNew frequency features:")
print([
    f"{col}_freq"
    for col in frequency_features
])


Feature: DeviceInfo
Training categories: 1640
Minimum frequency: 2.1167067429810006e-06
Maximum frequency: 0.7893114776306431

Top 5 frequencies:
DeviceInfo
__MISSING__    0.789311
Windows        0.083993
iOS Device     0.036160
MacOS          0.022951
Trident/7.0    0.013659

Feature: id_33
Training categories: 213
Minimum frequency: 2.1167067429810006e-06
Maximum frequency: 0.8691600060961154

Top 5 frequencies:
id_33
__MISSING__    0.869160
1920x1080      0.030608
1366x768       0.014116
1334x750       0.011858
2208x1242      0.008818

Frequency encoding completed.

New frequency features:
['DeviceInfo_freq', 'id_33_freq']


## Step 24C — Training-Based Rare-Category Grouping

Moderate-cardinality categorical variables `id_30` and `id_31` contain
categories that occur only a small number of times.

To reduce sparse and unstable categories, values appearing fewer than 10 times
in the training data will be grouped into a common `__RARE__` category.

The frequency threshold is determined using the training period only.

The same category grouping learned from training will then be applied to the
validation period.

Missing values are preserved separately as `__MISSING__`.

In [31]:
# Step 24C — Training-based rare-category grouping

rare_threshold = 10

rare_category_maps = {}

for col in rare_group_features:

    # Treat missing values separately
    train_values = train_df[col].fillna("__MISSING__")

    # Calculate category frequencies using training data only
    frequencies = train_values.value_counts()

    # Identify categories below the threshold
    rare_categories = frequencies[
        frequencies < rare_threshold
    ].index

    rare_category_maps[col] = set(rare_categories)

    print("\n" + "=" * 60)
    print(f"Feature: {col}")
    print("=" * 60)

    print("Total training categories:", len(frequencies))
    print("Rare categories:", len(rare_categories))
    print("Categories retained:", len(frequencies) - len(rare_categories))


# Apply the training-derived grouping to train and validation

for col in rare_group_features:

    train_values = train_df[col].fillna("__MISSING__")
    valid_values = valid_df[col].fillna("__MISSING__")

    train_df[f"{col}_grouped"] = train_values.apply(
        lambda x: "__RARE__"
        if x in rare_category_maps[col]
        else x
    )

    valid_df[f"{col}_grouped"] = valid_values.apply(
        lambda x: "__RARE__"
        if x in rare_category_maps[col]
        else x
    )


print("\nRare-category grouping completed.")

print("\nNew grouped features:")
print([
    f"{col}_grouped"
    for col in rare_group_features
])


Feature: id_31
Total training categories: 111
Rare categories: 25
Categories retained: 86

Feature: id_30
Total training categories: 73
Rare categories: 3
Categories retained: 70

Rare-category grouping completed.

New grouped features:
['id_31_grouped', 'id_30_grouped']


## Step 24D — Prepare the One-Hot Encoded Features

In [32]:
handle_unknown="ignore"

Categorical variables with manageable cardinality are converted into binary
indicator features using one-hot encoding.

The grouped versions of `id_30` and `id_31` are used after rare-category
grouping.

Missing values are retained as explicit categories.

The encoder is fitted on the training data only.

`handle_unknown="ignore"` ensures that categories appearing in the validation
period but not in the training period do not cause preprocessing errors.

The original high-cardinality variables `DeviceInfo` and `id_33` are excluded
from one-hot encoding because their frequency-encoded representations are
used instead.

In [33]:
# Step 24D — Prepare categorical data for one-hot encoding

from sklearn.preprocessing import OneHotEncoder

# Use grouped versions for id_30 and id_31
grouped_categorical_features = [
    f"{col}_grouped"
    for col in rare_group_features
]

# Remaining categorical features
remaining_onehot_features = onehot_features

# Complete one-hot feature list
final_onehot_features = (
    grouped_categorical_features
    + remaining_onehot_features
)

print("Total one-hot input features:", len(final_onehot_features))

print("\nGrouped categorical features:")
print(grouped_categorical_features)

print("\nOther categorical features:")
print(remaining_onehot_features)


# Create the encoder
onehot_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

# Fit ONLY on training data
onehot_encoder.fit(
    train_df[final_onehot_features]
)

print("\nOne-hot encoder fitted successfully.")

print(
    "Generated one-hot features:",
    len(onehot_encoder.get_feature_names_out())
)

Total one-hot input features: 29

Grouped categorical features:
['id_31_grouped', 'id_30_grouped']

Other categorical features:
['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType']

One-hot encoder fitted successfully.
Generated one-hot features: 365


# Step 24E — Numerical preprocessing

In [34]:
# Step 24E — Numerical preprocessing

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Near-constant numerical features identified during the audit
near_constant_features = [
    "V1",
    "V14",
    "V41",
    "V65",
    "V88",
    "V107",
    "V305"
]

# Remove near-constant features from the numerical feature list
final_numerical_features = [
    col
    for col in final_numerical_features
    if col not in near_constant_features
]

print("Numerical features before near-constant removal:", 409)
print("Near-constant features removed:", len(near_constant_features))
print("Final numerical features:", len(final_numerical_features))

print("\nRemoved features:")
print(near_constant_features)

# Create numerical preprocessing objects
numerical_imputer = SimpleImputer(
    strategy="median"
)

numerical_scaler = StandardScaler()

# Fit ONLY on training data
numerical_imputer.fit(
    train_df[final_numerical_features]
)

# Transform training and validation data
train_numerical_imputed = numerical_imputer.transform(
    train_df[final_numerical_features]
)

valid_numerical_imputed = numerical_imputer.transform(
    valid_df[final_numerical_features]
)

# Scale numerical features
train_numerical_scaled = numerical_scaler.fit_transform(
    train_numerical_imputed
)

valid_numerical_scaled = numerical_scaler.transform(
    valid_numerical_imputed
)

print("\nNumerical imputation completed.")
print("Training numerical matrix shape:", train_numerical_scaled.shape)
print("Validation numerical matrix shape:", valid_numerical_scaled.shape)

print(
    "\nRemaining NaN values in training:",
    np.isnan(train_numerical_scaled).sum()
)

print(
    "Remaining NaN values in validation:",
    np.isnan(valid_numerical_scaled).sum()
)

Numerical features before near-constant removal: 409
Near-constant features removed: 7
Final numerical features: 402

Removed features:
['V1', 'V14', 'V41', 'V65', 'V88', 'V107', 'V305']

Numerical imputation completed.
Training numerical matrix shape: (472432, 402)
Validation numerical matrix shape: (118108, 402)

Remaining NaN values in training: 0
Remaining NaN values in validation: 0


# Step 24F — Build the Final Modeling Matrix

In [35]:
# Step 24F — Prepare final modeling feature groups
# tqdm is used to show progress during feature preparation.

# ---------------------------------------------------------
# 1. Frequency-encoded features
# ---------------------------------------------------------

frequency_feature_names = []

for col in tqdm(
    frequency_features,
    desc="Preparing frequency features"
):
    frequency_feature_names.append(f"{col}_freq")


# ---------------------------------------------------------
# 2. Final numerical features
# ---------------------------------------------------------

final_model_numerical_features = []

for col in tqdm(
    final_numerical_features,
    desc="Preparing numerical features"
):
    final_model_numerical_features.append(col)

final_model_numerical_features.extend(
    frequency_feature_names
)


# ---------------------------------------------------------
# 3. Final categorical features
# ---------------------------------------------------------

final_model_categorical_features = []

for col in tqdm(
    final_onehot_features,
    desc="Preparing categorical features"
):
    final_model_categorical_features.append(col)


# ---------------------------------------------------------
# 4. Summary
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("FINAL MODEL INPUTS")
print("=" * 60)

print(
    "Numerical inputs:",
    len(final_model_numerical_features)
)

print(
    "Categorical inputs:",
    len(final_model_categorical_features)
)

print(
    "Total preprocessing inputs:",
    len(final_model_numerical_features)
    + len(final_model_categorical_features)
)

print("\nFrequency features:")
print(frequency_feature_names)

print("\nCategorical features:")
print(final_model_categorical_features)

Preparing categorical features: 100%|██████████| 29/29 [00:00<00:00, 298857.04it/s]


FINAL MODEL INPUTS
Numerical inputs: 404
Categorical inputs: 29
Total preprocessing inputs: 433

Frequency features:
['DeviceInfo_freq', 'id_33_freq']

Categorical features:
['id_31_grouped', 'id_30_grouped', 'ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType']


## Step 24G — Preprocessing Pipeline

The preprocessing pipeline combines numerical and categorical transformations
into a single reproducible workflow.

Numerical features are median-imputed and standardized.

Categorical features are imputed with an explicit missing category and then
one-hot encoded. Unknown categories in future data are ignored safely.

The preprocessing object is fitted using training data only.

Before transforming the complete dataset, the pipeline will be tested on a
small training sample to verify the resulting feature representation without
creating an unnecessary large memory allocation.

In [36]:
# Step 24G — Build the preprocessing pipeline

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# ---------------------------------------------------------
# Numerical preprocessing
# ---------------------------------------------------------

numerical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)


# ---------------------------------------------------------
# Categorical preprocessing
# ---------------------------------------------------------

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="__MISSING__"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True,
                dtype=np.float32
            )
        )
    ]
)


# ---------------------------------------------------------
# Combined preprocessing
# ---------------------------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_pipeline,
            final_model_numerical_features
        ),
        (
            "categorical",
            categorical_pipeline,
            final_model_categorical_features
        )
    ],
    remainder="drop"
)


print("Preprocessing pipeline created.")
print("Numerical inputs:", len(final_model_numerical_features))
print("Categorical inputs:", len(final_model_categorical_features))

Preprocessing pipeline created.
Numerical inputs: 404
Categorical inputs: 29


In [37]:
# Step 24H — Test preprocessing on a small sample

from tqdm import tqdm

sample_size = 10_000

print("Selecting training sample...")

train_sample = train_df[
    final_model_numerical_features
    + final_model_categorical_features
].iloc[:sample_size].copy()

print("Sample shape:", train_sample.shape)

print("\nFitting preprocessing pipeline on sample...")

preprocessor.fit(train_sample)

print("Pipeline fitted successfully.")

print("\nTransforming sample...")

X_sample = preprocessor.transform(train_sample)

print("Transformation completed.")

print("\n" + "=" * 60)
print("PREPROCESSING TEST RESULTS")
print("=" * 60)

print("Input rows:", train_sample.shape[0])
print("Input features:", train_sample.shape[1])
print("Output shape:", X_sample.shape)
print("Output type:", type(X_sample).__name__)
print("Output dtype:", X_sample.dtype)

Selecting training sample...
Sample shape: (10000, 433)

Fitting preprocessing pipeline on sample...
Pipeline fitted successfully.

Transforming sample...
Transformation completed.

PREPROCESSING TEST RESULTS
Input rows: 10000
Input features: 433
Output shape: (10000, 682)
Output type: ndarray
Output dtype: float64


## Step 24I — Memory-Efficient Preprocessing Output

The initial preprocessing test successfully transformed the sample, but the
result was returned as a dense float64 array.

For the full fraud dataset, a dense representation would consume substantial
memory because the dataset contains hundreds of thousands of transactions.

The preprocessing pipeline will therefore be configured to preserve sparse
output where appropriate and use float32 values.

The pipeline will be retested on the same small sample before processing the
full training and validation periods.

In [38]:
# Step 24I — Rebuild preprocessing pipeline with memory-efficient output

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# ---------------------------------------------------------
# Numerical preprocessing
# ---------------------------------------------------------

numerical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)


# ---------------------------------------------------------
# Categorical preprocessing
# ---------------------------------------------------------

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="__MISSING__"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True,
                dtype=np.float32
            )
        )
    ]
)


# ---------------------------------------------------------
# Combined preprocessing
# ---------------------------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_pipeline,
            final_model_numerical_features
        ),
        (
            "categorical",
            categorical_pipeline,
            final_model_categorical_features
        )
    ],
    remainder="drop",
    sparse_threshold=1.0
)


print("Memory-efficient preprocessing pipeline created.")
print("Numerical inputs:", len(final_model_numerical_features))
print("Categorical inputs:", len(final_model_categorical_features))

Memory-efficient preprocessing pipeline created.
Numerical inputs: 404
Categorical inputs: 29


In [39]:
# Step 24I — Test sparse preprocessing output

print("Fitting preprocessing pipeline...")

preprocessor.fit(train_sample)

print("Pipeline fitted.")

print("\nTransforming sample...")

X_sample = preprocessor.transform(train_sample)

print("Transformation completed.")

print("\n" + "=" * 60)
print("MEMORY-EFFICIENT PREPROCESSING TEST")
print("=" * 60)

print("Input shape:", train_sample.shape)
print("Output shape:", X_sample.shape)
print("Output type:", type(X_sample).__name__)
print("Output dtype:", X_sample.dtype)

if hasattr(X_sample, "nnz"):
    print("Non-zero values:", X_sample.nnz)
    print("Total values:", X_sample.shape[0] * X_sample.shape[1])
    print(
        "Sparsity:",
        round(
            1 - X_sample.nnz /
            (X_sample.shape[0] * X_sample.shape[1]),
            4
        )
    )

Fitting preprocessing pipeline...
Pipeline fitted.

Transforming sample...
Transformation completed.

MEMORY-EFFICIENT PREPROCESSING TEST
Input shape: (10000, 433)
Output shape: (10000, 682)
Output type: csr_matrix
Output dtype: float64
Non-zero values: 4300058
Total values: 6820000
Sparsity: 0.3695


## Step 24J — Final Preprocessing Pipeline

The preprocessing strategy has now been validated on a representative sample.

The final pipeline combines:

- Median imputation for numerical features
- Standardization of numerical features
- Explicit missing-value handling for categorical features
- One-hot encoding for manageable categorical variables
- Training-derived frequency encoding for high-cardinality variables
- Unknown-category handling for future validation data

The preprocessing object will be fitted using the training period only.

The transformed data will remain sparse to reduce memory usage.

The preprocessing pipeline will be reused consistently during model training
and validation to prevent inconsistent transformations.

In [40]:
# Step 24J — Final preprocessing pipeline

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import numpy as np

# ---------------------------------------------------------
# Numerical preprocessing
# ---------------------------------------------------------

numerical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)


# ---------------------------------------------------------
# Categorical preprocessing
# ---------------------------------------------------------

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="__MISSING__"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True,
                dtype=np.float32
            )
        )
    ]
)


# ---------------------------------------------------------
# Combined preprocessing
# ---------------------------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_pipeline,
            final_model_numerical_features
        ),
        (
            "categorical",
            categorical_pipeline,
            final_model_categorical_features
        )
    ],
    remainder="drop",
    sparse_threshold=1.0
)


print("=" * 60)
print("FINAL PREPROCESSING PIPELINE")
print("=" * 60)

print("Numerical inputs:", len(final_model_numerical_features))
print("Categorical inputs:", len(final_model_categorical_features))
print(
    "Total preprocessing inputs:",
    len(final_model_numerical_features)
    + len(final_model_categorical_features)
)

print("\nPipeline ready.")

FINAL PREPROCESSING PIPELINE
Numerical inputs: 404
Categorical inputs: 29
Total preprocessing inputs: 433

Pipeline ready.


In [41]:
# Step 25A — Prepare baseline data without duplicating the full dataset

y_train = train_df["isFraud"].to_numpy(dtype=np.int8)
y_valid = valid_df["isFraud"].to_numpy(dtype=np.int8)

print("=" * 60)
print("BASELINE MODEL DATA")
print("=" * 60)

print("Training rows:", len(train_df))
print("Validation rows:", len(valid_df))

print("Training fraud rate:", y_train.mean())
print("Validation fraud rate:", y_valid.mean())

print("\nTraining input features:", len(final_model_numerical_features)
      + len(final_model_categorical_features))

print("Numerical inputs:", len(final_model_numerical_features))
print("Categorical inputs:", len(final_model_categorical_features))

BASELINE MODEL DATA
Training rows: 472432
Validation rows: 118108
Training fraud rate: 0.03513521522674162
Validation fraud rate: 0.034409184813899145

Training input features: 433
Numerical inputs: 404
Categorical inputs: 29


In [42]:
# Step 25A — Remove large temporary preprocessing objects

import gc

temporary_objects = [
    "train_numerical_imputed",
    "valid_numerical_imputed",
    "train_numerical_scaled",
    "valid_numerical_scaled",
    "train_onehot",
    "valid_onehot",
    "train_frequency",
    "valid_frequency",
    "train_numerical_sparse",
    "valid_numerical_sparse",
    "train_frequency_sparse",
    "valid_frequency_sparse",
    "X_train",
    "X_valid",
    "X_sample"
]

removed = []

for name in temporary_objects:
    if name in globals():
        del globals()[name]
        removed.append(name)

gc.collect()

print("=" * 60)
print("MEMORY CLEANUP")
print("=" * 60)

print("Temporary objects removed:", len(removed))

if removed:
    print("\nRemoved:")
    print(removed)

print("\nGarbage collection completed.")

MEMORY CLEANUP
Temporary objects removed: 5

Removed:
['train_numerical_imputed', 'valid_numerical_imputed', 'train_numerical_scaled', 'valid_numerical_scaled', 'X_sample']

Garbage collection completed.


## Step 25B — Target Preparation

The target variable is `isFraud`.

The model will learn from the training period and will later be evaluated only
on the chronologically later validation period.

The target is converted to a compact integer representation to reduce memory
usage.

In [43]:
# Step 25B — Prepare target arrays

y_train = train_df["isFraud"].to_numpy(dtype=np.int8)
y_valid = valid_df["isFraud"].to_numpy(dtype=np.int8)

print("=" * 60)
print("TARGET PREPARATION")
print("=" * 60)

print("Training rows:", len(y_train))
print("Validation rows:", len(y_valid))

print("Training fraud rate:", round(y_train.mean(), 6))
print("Validation fraud rate:", round(y_valid.mean(), 6))

print("\nTraining fraud cases:", int(y_train.sum()))
print("Training legitimate cases:", int((y_train == 0).sum()))

print("Validation fraud cases:", int(y_valid.sum()))
print("Validation legitimate cases:", int((y_valid == 0).sum()))

TARGET PREPARATION
Training rows: 472432
Validation rows: 118108
Training fraud rate: 0.035135
Validation fraud rate: 0.034409

Training fraud cases: 16599
Training legitimate cases: 455833
Validation fraud cases: 4064
Validation legitimate cases: 114044


# Step 26 — Save Final Train/Validation Data

In [45]:
# Step 24K — Finalize and save the actual modelling datasets

import os
import gc
import pandas as pd
import numpy as np

processed_dir = "../data/processed"
os.makedirs(processed_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. Recreate training-only frequency maps
# ------------------------------------------------------------

def build_frequency_map(series):
    values = series.fillna("__MISSING__")
    return values.value_counts(normalize=True)


deviceinfo_freq_map = build_frequency_map(train_df["DeviceInfo"])
id33_freq_map = build_frequency_map(train_df["id_33"])

# Apply frequency encoding
train_df["DeviceInfo_freq"] = (
    train_df["DeviceInfo"]
    .fillna("__MISSING__")
    .map(deviceinfo_freq_map)
    .fillna(0)
)

valid_df["DeviceInfo_freq"] = (
    valid_df["DeviceInfo"]
    .fillna("__MISSING__")
    .map(deviceinfo_freq_map)
    .fillna(0)
)

train_df["id_33_freq"] = (
    train_df["id_33"]
    .fillna("__MISSING__")
    .map(id33_freq_map)
    .fillna(0)
)

valid_df["id_33_freq"] = (
    valid_df["id_33"]
    .fillna("__MISSING__")
    .map(id33_freq_map)
    .fillna(0)
)

# ------------------------------------------------------------
# 2. Recreate training-only rare-category maps
# ------------------------------------------------------------

def build_rare_map(series, threshold=10):
    values = series.fillna("__MISSING__")
    counts = values.value_counts()

    return {
        category: category if count > threshold else "__RARE__"
        for category, count in counts.items()
    }


id31_map = build_rare_map(train_df["id_31"])
id30_map = build_rare_map(train_df["id_30"])

# Apply rare grouping
train_df["id_31_grouped"] = (
    train_df["id_31"]
    .fillna("__MISSING__")
    .map(id31_map)
    .fillna("__RARE__")
)

valid_df["id_31_grouped"] = (
    valid_df["id_31"]
    .fillna("__MISSING__")
    .map(id31_map)
    .fillna("__RARE__")
)

train_df["id_30_grouped"] = (
    train_df["id_30"]
    .fillna("__MISSING__")
    .map(id30_map)
    .fillna("__RARE__")
)

valid_df["id_30_grouped"] = (
    valid_df["id_30"]
    .fillna("__MISSING__")
    .map(id30_map)
    .fillna("__RARE__")
)

# ------------------------------------------------------------
# 3. Define final modelling feature groups
# ------------------------------------------------------------

near_constant_features = [
    "V1",
    "V14",
    "V41",
    "V65",
    "V88",
    "V107",
    "V305"
]

redundant_features = [
    "D12"
]

frequency_features = [
    "DeviceInfo_freq",
    "id_33_freq"
]

grouped_features = [
    "id_31_grouped",
    "id_30_grouped"
]

# Original categorical columns represented by engineered versions
categorical_to_replace = [
    "DeviceInfo",
    "id_33",
    "id_31",
    "id_30"
]

# Start from the current modelling feature inventory
base_features = [
    col for col in model_features
    if col not in near_constant_features
    and col not in redundant_features
    and col not in categorical_to_replace
]

# Add engineered categorical representations
final_model_features = (
    base_features
    + frequency_features
    + grouped_features
)

# Remove accidental duplicates while preserving order
final_model_features = list(dict.fromkeys(final_model_features))

# ------------------------------------------------------------
# 4. Verify final feature inventory
# ------------------------------------------------------------

missing_train = [
    col for col in final_model_features
    if col not in train_df.columns
]

missing_valid = [
    col for col in final_model_features
    if col not in valid_df.columns
]

print("=" * 60)
print("FINAL MODELLING DATASET")
print("=" * 60)

print("\nFinal feature count:", len(final_model_features))

print("Missing features in training:", len(missing_train))
print("Missing features in validation:", len(missing_valid))

if missing_train:
    print("\nMissing training features:")
    print(missing_train)

if missing_valid:
    print("\nMissing validation features:")
    print(missing_valid)

# ------------------------------------------------------------
# 5. Save only final modelling features + target
# ------------------------------------------------------------

save_columns = final_model_features + [target]

train_to_save = train_df[save_columns]
valid_to_save = valid_df[save_columns]

train_path = os.path.join(
    processed_dir,
    "train_features.csv"
)

valid_path = os.path.join(
    processed_dir,
    "validation_features.csv"
)

print("\nTraining shape to save:", train_to_save.shape)
print("Validation shape to save:", valid_to_save.shape)

train_to_save.to_csv(
    train_path,
    index=False
)

valid_to_save.to_csv(
    valid_path,
    index=False
)

print("\nFiles saved:")
print(train_path)
print(valid_path)

print("\n" + "=" * 60)
print("FINAL MODELLING DATASET SAVED")
print("=" * 60)

# Free temporary references
del train_to_save
del valid_to_save
gc.collect()

FINAL MODELLING DATASET

Final feature count: 433
Missing features in training: 0
Missing features in validation: 0

Training shape to save: (472432, 434)
Validation shape to save: (118108, 434)

Files saved:
../data/processed/train_features.csv
../data/processed/validation_features.csv

FINAL MODELLING DATASET SAVED


0